# Week 03B — Convolution, Filtering, Fourier and Morphology

**MCTA 4364 Machine Vision** · Session 2 of Week 3 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W03_preprocessing/W03B_filtering_fourier_morphology.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W03_preprocessing/W03B_filtering_fourier_morphology.ipynb)

Point operations look at one pixel at a time. **Neighbourhood operations** compute each output pixel from a small window around it.
They remove noise, find edges, sharpen detail and clean up binary masks. The central tool, **convolution**, is also the building block
of the convolutional neural networks you will meet in Week 6, where the kernels are *learned* instead of designed.

### Learning outcomes
By the end of this session you will be able to:
1. Compute a **convolution / correlation** by hand and explain kernels, borders and **separability**.
2. Model common **noise** types and choose the best **smoothing** filter using PSNR (mean, Gaussian, median, bilateral, non-local means).
3. Compute image **derivatives** (Sobel, Laplacian), gradient magnitude and orientation, and **sharpen** with unsharp masking.
4. Interpret the **2D Fourier transform**, filter in the frequency domain and remove **periodic noise**.
5. Apply **morphological** operations (erode, dilate, open, close, gradient, top-hat, skeleton) to clean and analyse binary images.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–15 | 1–2 | Convolution, borders, separable Gaussian |
| 15–30 | 3 | Noise and restoration benchmark (interactive) |
| 30–42 | 4 | Derivatives, edges, sharpening |
| 42–57 | 5 | The Fourier view: spectra, filtering, notch filter |
| 57–72 | 6 | Morphology and a cleaning pipeline |
| 72–80 | 9 | Quiz and exit ticket |
| Home | 7–8 | Exercises with self-checks, challenge |

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import skimage.data
from skimage.morphology import skeletonize

from cvhelpers import SMOKE, check, interact, load_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
cam = skimage.data.camera()
coins = skimage.data.coins()
sudoku = load_sample("sudoku.png", gray=True)
rng = np.random.default_rng(0)

## 1. Convolution: the neighbourhood operation

A linear filter replaces each pixel by a **weighted sum** of its neighbours. The weights form a small matrix, the **kernel** $h$:

$$ g(x, y) = \sum_{i=-k}^{k}\sum_{j=-k}^{k} h(i, j)\, f(x + i,\, y + j) \qquad \text{(correlation, what } \texttt{cv2.filter2D} \text{ computes)} $$

True **convolution** flips the kernel, $f(x - i, y - j)$. The two are identical for symmetric kernels (box, Gaussian, Laplacian) and differ
only by a sign or mirror for others (Sobel). Convolution is **linear** and **shift-invariant**, it commutes and it is associative
($(f * h_1) * h_2 = f * (h_1 * h_2)$), so several filters can be merged into one kernel.

In [ ]:
f_small = np.array([[10, 10, 10, 80, 80], [10, 10, 10, 80, 80], [10, 10, 10, 80, 80], [10, 10, 10, 80, 80], [10, 10, 10, 80, 80]], np.float32)
kernel = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], np.float32) / 16
out = cv2.filter2D(f_small, -1, kernel, borderType=cv2.BORDER_REPLICATE)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, mat, title, fmt in [(axes[0], f_small, "input f (a vertical edge)", "{:.0f}"), (axes[1], kernel * 16, "kernel h (x 1/16)", "{:.0f}"),
                            (axes[2], out, "output g = filter2D(f, h)", "{:.1f}")]:
    ax.imshow(mat, cmap="Blues", vmin=0, vmax=max(90, mat.max()))
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=12)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
axes[0].add_patch(plt.Rectangle((1.5, 0.5), 3, 3, fill=False, ec="#d93025", lw=3))
axes[2].add_patch(plt.Rectangle((2.5, 1.5), 1, 1, fill=False, ec="#d93025", lw=3))
plt.suptitle("Figure 1.1 - Each output pixel is the kernel-weighted sum of the red 3x3 window: the edge gets smoothed", fontweight="bold")
plt.tight_layout(); plt.show()
manual = (f_small[1:4, 2:5] * kernel).sum()
print(f"manual sum at the red window: {manual:.2f} | filter2D: {out[2, 3]:.2f}")

### Borders
Near the image border the window hangs over the edge, so values must be invented. The choice affects the outermost pixels, and matters for
measurements near the border and for large kernels.

In [ ]:
tiny = cv2.resize(cam[180:260, 200:280], (40, 40), interpolation=cv2.INTER_AREA)
modes = {"BORDER_CONSTANT (zeros)": cv2.BORDER_CONSTANT, "BORDER_REPLICATE (aaa|abc)": cv2.BORDER_REPLICATE,
         "BORDER_REFLECT_101 (default, cb|abc)": cv2.BORDER_REFLECT_101, "BORDER_WRAP (bc|abc, periodic)": cv2.BORDER_WRAP}
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, mode) in zip(axes, modes.items()):
    padded = cv2.copyMakeBorder(tiny, 15, 15, 15, 15, mode, value=0)
    ax.imshow(padded, cmap="gray"); ax.add_patch(plt.Rectangle((14.5, 14.5), 40, 40, fill=False, ec="#d93025", lw=2))
    ax.set_title(name, fontsize=10); ax.axis("off")
plt.suptitle("Figure 1.2 - Border modes: how the image is extended beyond its edges (red = real image)", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Smoothing and separability

- **Box (mean) filter:** all weights equal. Simple, but it has sharp edges in its weights, which cause ripple artefacts.
- **Gaussian filter:** $h(x, y) = \frac{1}{2\pi\sigma^2}e^{-(x^2+y^2)/(2\sigma^2)}$. Weights fall smoothly; $\sigma$ sets the amount of blur, and a kernel of width
  about $6\sigma$ captures it fully.

**Separability.** The 2D Gaussian is the product of two 1D Gaussians, $h(x, y) = h(x)\,h(y)$, so filtering with a $k \times k$ kernel can be done as a
1D horizontal pass followed by a 1D vertical pass: $2k$ multiplications per pixel instead of $k^2$.

In [ ]:
fig = plt.figure(figsize=(18, 4.6))
for k, (name, K2) in enumerate([("box 9x9", np.ones((9, 9)) / 81),
                                ("Gaussian sigma = 1.5", cv2.getGaussianKernel(9, 1.5) @ cv2.getGaussianKernel(9, 1.5).T)]):
    ax = fig.add_subplot(1, 4, k + 1, projection="3d")
    X, Y = np.meshgrid(range(9), range(9))
    ax.plot_surface(X, Y, K2, cmap="viridis"); ax.set_title(name); ax.set_zticks([])
for k, (name, im) in enumerate([("box 9x9", cv2.blur(cam, (9, 9))), ("Gaussian sigma = 1.5 (9x9)", cv2.GaussianBlur(cam, (9, 9), 1.5))]):
    ax = fig.add_subplot(1, 4, k + 3); ax.imshow(im[150:350, 150:350], cmap="gray"); ax.set_title(name); ax.axis("off")
plt.suptitle("Figure 2.1 - Box vs Gaussian kernels and their effect", fontweight="bold")
plt.tight_layout(); plt.show()

big = cv2.resize(cam, (2000, 2000)).astype(np.float32)
g1 = cv2.getGaussianKernel(31, 5)
t0 = time.perf_counter(); full = cv2.filter2D(big, -1, g1 @ g1.T); t_full = time.perf_counter() - t0
t0 = time.perf_counter(); sep = cv2.sepFilter2D(big, -1, g1, g1); t_sep = time.perf_counter() - t0
print(f"31x31 kernel on 2000x2000: 2D kernel {t_full * 1000:.0f} ms | separable {t_sep * 1000:.0f} ms | "
      f"max difference {np.abs(full - sep).max():.2e}")

## 3. Noise and restoration

| Noise | Model | Typical source | Best simple filter |
|---|---|---|---|
| **Gaussian** | $f = f_0 + n,\ n \sim \mathcal{N}(0, \sigma^2)$ | sensor electronics, heat | Gaussian, bilateral, non-local means |
| **Salt-and-pepper** | random pixels set to 0 or 255 | dead/hot pixels, transmission errors | **median** |
| **Speckle** | $f = f_0 (1 + n)$ | laser, ultrasound, radar | median, log + Gaussian |
| **Shot (Poisson)** | variance ∝ brightness | photon counting in low light | averaging frames, non-local means |

**Non-linear filters:** the **median** takes the middle value of the window, so isolated extreme pixels are simply ignored and edges are preserved.
The **bilateral** filter weights neighbours by both *distance* and *intensity similarity*: pixels across an edge (very different intensity) get
almost no weight, so the edge survives. **Non-local means** averages similar *patches* anywhere in a search window.

We measure quality with the **peak signal-to-noise ratio**, $\text{PSNR} = 10\log_{10}\frac{255^2}{\text{MSE}}$ dB (higher is better).

In [ ]:
clean = cam
noisy = {
    "Gaussian (sigma 20)": np.clip(clean + rng.normal(0, 20, clean.shape), 0, 255).astype(np.uint8),
    "salt & pepper (5%)": clean.copy(),
    "speckle (0.2)": np.clip(clean * (1 + rng.normal(0, 0.2, clean.shape)), 0, 255).astype(np.uint8),
    "Poisson (low light)": np.clip(rng.poisson(clean / 8.0) * 8.0, 0, 255).astype(np.uint8)}
mask = rng.random(clean.shape)
noisy["salt & pepper (5%)"][mask < 0.025] = 0
noisy["salt & pepper (5%)"][mask > 0.975] = 255
filters = {"none": lambda x: x, "mean 5x5": lambda x: cv2.blur(x, (5, 5)), "Gaussian s=1.2": lambda x: cv2.GaussianBlur(x, (0, 0), 1.2),
           "median 5": lambda x: cv2.medianBlur(x, 5), "bilateral": lambda x: cv2.bilateralFilter(x, 9, 50, 5),
           "non-local means": lambda x: cv2.fastNlMeansDenoising(x, None, 18, 7, 21)}
fig, axes = plt.subplots(2, 4, figsize=(21, 9))
for j, (name, im) in enumerate(noisy.items()):
    axes[0, j].imshow(im[150:350, 150:350], cmap="gray"); axes[0, j].set_title(f"{name}: {cv2.PSNR(clean, im):.1f} dB"); axes[0, j].axis("off")
    axes[1, j].hist((im.astype(int) - clean.astype(int)).ravel(), bins=100, range=(-120, 120), color="#1a73e8")
    axes[1, j].set_yscale("log"); axes[1, j].set_title("noise histogram (noisy - clean)")
plt.suptitle("Figure 3.1 - Four noise models (zoomed) and their error distributions", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
psnr = np.array([[cv2.PSNR(clean, fn(im)) for fn in filters.values()] for im in noisy.values()])
fig, ax = plt.subplots(figsize=(12, 4.3))
im_ = ax.imshow(psnr, cmap="viridis")
ax.set_xticks(range(len(filters)), list(filters), rotation=20, ha="right"); ax.set_yticks(range(len(noisy)), list(noisy))
for (i, j), v in np.ndenumerate(psnr):
    best = j == psnr[i].argmax()
    ax.text(j, i, f"{v:.1f}", ha="center", va="center", color="k" if v > psnr.mean() else "w", fontweight="bold" if best else None,
            bbox=dict(fc="none", ec="#fbbc04", lw=2) if best else None)
plt.colorbar(im_, label="PSNR (dB)")
ax.set_title("Figure 3.2 - Restoration benchmark: PSNR of every filter on every noise type (best boxed)", fontweight="bold")
plt.show()

In [ ]:
# Figure 3.3: why the bilateral filter preserves edges - the weights at one pixel next to an edge
edges_mag = cv2.magnitude(cv2.Sobel(cv2.GaussianBlur(cam, (0, 0), 3), cv2.CV_32F, 1, 0), cv2.Sobel(cv2.GaussianBlur(cam, (0, 0), 3), cv2.CV_32F, 0, 1))
ey, ex = np.unravel_index(edges_mag[100:400, 100:400].argmax(), (300, 300)); ey += 100; ex += 100   # strongest edge pixel
patch = noisy["Gaussian (sigma 20)"][ey - 10:ey + 11, ex - 10:ex + 11].astype(float)
c = patch[10, 10]
yy, xx = np.mgrid[-10:11, -10:11]
w_space = np.exp(-(xx ** 2 + yy ** 2) / (2 * 4.0 ** 2))
w_range = np.exp(-((patch - c) ** 2) / (2 * 40.0 ** 2))
fig, axes = plt.subplots(1, 4, figsize=(19, 4.3))
for ax, im, title in zip(axes, [patch, w_space, w_range, w_space * w_range],
                         ["21x21 patch (centre = red)", "spatial weight (Gaussian)", "range weight (similar intensity)", "bilateral = product"]):
    ax.imshow(im, cmap="gray" if title.startswith("21") else "magma"); ax.set_title(title); ax.axis("off")
    ax.plot(10, 10, "o", mfc="none", mec="#d93025", ms=10, mew=2)
plt.suptitle("Figure 3.3 - Bilateral filter weights: neighbours across an edge are ignored", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: filter explorer

In [ ]:
def filter_demo(noise="salt & pepper (5%)", method="median", size=5):
    im = noisy[noise]
    k = size if size % 2 else size + 1
    out = {"mean": cv2.blur(im, (k, k)), "Gaussian": cv2.GaussianBlur(im, (k, k), 0), "median": cv2.medianBlur(im, k),
           "bilateral": cv2.bilateralFilter(im, k, 50, k)}[method]
    show(im[100:400, 100:400], out[100:400, 100:400], titles=[f"{noise}: {cv2.PSNR(clean, im):.1f} dB",
                                                         f"{method} {k}: {cv2.PSNR(clean, out):.1f} dB"], figsize=(12, 6))


interact(filter_demo, noise=list(noisy), method=["mean", "Gaussian", "median", "bilateral"],
         size=widgets.IntSlider(value=5, min=3, max=15, step=2))

## 4. Derivatives, edges and sharpening

Edges are places where intensity changes quickly, so they appear as large **derivatives**. On a pixel grid the derivative is approximated by
differences, e.g. $\partial f/\partial x \approx f(x+1) - f(x-1)$. The **Sobel** kernels combine this difference with smoothing in the other direction:

$$ S_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}, \quad S_y = S_x^\top, \qquad
|\nabla f| = \sqrt{G_x^2 + G_y^2}, \quad \theta = \operatorname{atan2}(G_y, G_x) $$

The **Laplacian** $\nabla^2 f = f_{xx} + f_{yy}$ (kernel $[[0,1,0],[1,-4,1],[0,1,0]]$) responds to second-order changes: it crosses zero at edges.
Derivatives **amplify noise**, so images are smoothed first (Gaussian derivative, or Laplacian of Gaussian, LoG). Week 4 builds the Canny edge detector on these ideas.

In [ ]:
row = cam[300].astype(float)
noisy_row = row + rng.normal(0, 8, row.shape)
fig, axes = plt.subplots(2, 1, figsize=(15, 5.2), sharex=True)
axes[0].plot(noisy_row, color="#9aa0a6", label="noisy row"); axes[0].plot(cv2.GaussianBlur(noisy_row[None], (0, 0), 3)[0], lw=2, label="smoothed")
axes[0].legend(); axes[0].set_ylabel("intensity")
axes[1].plot(np.gradient(noisy_row), color="#9aa0a6", label="derivative of noisy row")
axes[1].plot(np.gradient(cv2.GaussianBlur(noisy_row[None], (0, 0), 3)[0]), lw=2, color="#d93025", label="derivative of smoothed row")
axes[1].legend(); axes[1].set_ylabel("d/dx"); axes[1].set_xlabel("column")
plt.suptitle("Figure 4.1 - Derivatives amplify noise: smooth first, then differentiate", fontweight="bold")
plt.tight_layout(); plt.show()

gx = cv2.Sobel(cam, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(cam, cv2.CV_32F, 0, 1, ksize=3)
mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
hsv = np.dstack([(ang / 2).astype(np.uint8), np.full_like(cam, 255), cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)])
lap = cv2.Laplacian(cv2.GaussianBlur(cam, (0, 0), 1.5), cv2.CV_32F)
fig, axes = plt.subplots(1, 5, figsize=(24, 4.8))
for ax, im, title, cmap in [(axes[0], gx, "Gx (Sobel x)", "seismic"), (axes[1], gy, "Gy (Sobel y)", "seismic"),
                            (axes[2], mag, "gradient magnitude", "gray"), (axes[4], lap, "Laplacian of Gaussian", "seismic")]:
    v = np.abs(im).max() if cmap == "seismic" else None
    ax.imshow(im, cmap=cmap, vmin=-v if v else None, vmax=v); ax.set_title(title); ax.axis("off")
axes[3].imshow(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)); axes[3].set_title("orientation (colour) x magnitude (brightness)"); axes[3].axis("off")
plt.suptitle("Figure 4.2 - First and second derivatives of an image", fontweight="bold")
plt.tight_layout(); plt.show()

**Unsharp masking** sharpens by adding back the detail that blurring removes: $g = f + \lambda\,(f - f * G_\sigma)$.
(The name comes from darkroom photography, where a blurred "unsharp" negative was used as a mask.)

In [ ]:
def sharpen_demo(amount=1.0, sigma=2.0):
    blur = cv2.GaussianBlur(cam, (0, 0), sigma)
    sharp = cv2.addWeighted(cam, 1 + amount, blur, -amount, 0)
    show(cam[100:350, 180:430], sharp[100:350, 180:430], titles=["original", f"unsharp mask: amount {amount}, sigma {sigma}"], figsize=(12, 6))


interact(sharpen_demo, amount=widgets.FloatSlider(value=1.0, min=0, max=4, step=0.1), sigma=widgets.FloatSlider(value=2.0, min=0.5, max=8, step=0.5))

## 5. The Fourier view of filtering

Any image can be written as a sum of 2D sinusoidal **gratings** of different frequencies, orientations and amplitudes. The **2D discrete Fourier
transform (DFT)** computes those amplitudes and phases:

$$ F(u, v) = \sum_{x=0}^{W-1}\sum_{y=0}^{H-1} f(x, y)\, e^{-2\pi i\,(ux/W + vy/H)} $$

We display $\log(1 + |F|)$ with the zero frequency (the mean brightness) shifted to the centre. Points far from the centre are **high
frequencies** (fine detail, edges, noise); a point's direction from the centre is **perpendicular** to the stripes it represents.

The **convolution theorem** links the two worlds: $f * h \;\Leftrightarrow\; F \cdot H$. Convolution in space is multiplication in frequency. A
blur is a **low-pass** filter; an edge detector is a **high-pass** filter. Large kernels are often applied faster with the FFT.

In [ ]:
def spectrum(img):
    F = np.fft.fftshift(np.fft.fft2(img.astype(np.float32)))
    return np.log1p(np.abs(F))


n = 256
yy, xx = np.mgrid[0:n, 0:n]
gratings = {"vertical stripes (8 cycles)": np.sin(2 * np.pi * 8 * xx / n), "horizontal stripes (20 cycles)": np.sin(2 * np.pi * 20 * yy / n),
            "diagonal stripes": np.sin(2 * np.pi * (12 * xx + 12 * yy) / n),
            "two gratings summed": np.sin(2 * np.pi * 8 * xx / n) + np.sin(2 * np.pi * 30 * yy / n)}
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
for j, (name, g) in enumerate(gratings.items()):
    axes[0, j].imshow(g, cmap="gray"); axes[0, j].set_title(name, fontsize=9)
    axes[1, j].imshow(spectrum(g)[96:160, 96:160], cmap="magma", extent=[-32, 32, 32, -32]); axes[1, j].set_title("spectrum (zoomed centre)", fontsize=9)
for j, (name, im) in enumerate([("camera", cam), ("sudoku", cv2.resize(sudoku, (512, 512)))]):
    axes[0, 4 + j].imshow(im, cmap="gray"); axes[0, 4 + j].set_title(name)
    axes[1, 4 + j].imshow(spectrum(im), cmap="magma"); axes[1, 4 + j].set_title("log magnitude spectrum")
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Figure 5.1 - Every grating is a pair of dots in the spectrum; real images spread energy, mostly at low frequencies", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
F = np.fft.fftshift(np.fft.fft2(cam.astype(np.float32)))
R = np.hypot(*np.mgrid[-256:256, -256:256])
masks = {"ideal low-pass r=40": (R <= 40).astype(float), "Gaussian low-pass s=30": np.exp(-R ** 2 / (2 * 30 ** 2)),
         "Gaussian high-pass s=15": 1 - np.exp(-R ** 2 / (2 * 15 ** 2))}
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
for j, (name, Hm) in enumerate(masks.items()):
    out = np.real(np.fft.ifft2(np.fft.ifftshift(F * Hm)))
    axes[0, j].imshow(Hm, cmap="gray"); axes[0, j].set_title(f"mask H: {name}")
    axes[1, j].imshow(out if "high" not in name else np.abs(out), cmap="gray"); axes[1, j].set_title("result: " + ("ringing!" if "ideal" in name else
                                                                                                           "smooth blur" if "low" in name else "edges only"))
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Figure 5.2 - Filtering by multiplying the spectrum. Sharp (ideal) masks cause ringing; smooth (Gaussian) ones do not", fontweight="bold")
plt.tight_layout(); plt.show()

pad = 30        # the FFT treats the image as periodic, so pad with wrap-around before blurring in space
wrapped = cv2.copyMakeBorder(cam.astype(np.float32), pad, pad, pad, pad, cv2.BORDER_WRAP)
spatial = cv2.GaussianBlur(wrapped, (0, 0), 3)[pad:-pad, pad:-pad]
Hg = np.exp(-(2 * np.pi ** 2 * 9) * ((np.fft.fftfreq(512)[:, None]) ** 2 + (np.fft.fftfreq(512)[None, :]) ** 2))
freq = np.real(np.fft.ifft2(np.fft.fft2(cam.astype(np.float32)) * Hg))
print(f"Convolution theorem check - Gaussian blur in space vs multiplication in frequency: mean |difference| = "
      f"{np.abs(spatial - freq).mean():.3f} grey levels")

### Periodic noise removal (notch filter)
Electrical interference, vibrating fans or scanning artefacts add a **periodic pattern** to an image. In space it covers everything, but in the
spectrum it is just a few bright spikes, which a **notch filter** can zero out without touching the rest of the image. This is almost impossible
with spatial filters.

In [ ]:
YY, XX = np.mgrid[0:512, 0:512]
interference = 40 * np.sin(2 * np.pi * (60 * XX + 25 * YY) / 512)
corrupted = np.clip(cam + interference, 0, 255)
Fc = np.fft.fftshift(np.fft.fft2(corrupted))
power = np.log1p(np.abs(Fc))
notch = np.ones_like(power)
peaks = np.argwhere((power > np.percentile(power, 99.995)) & (R > 10))
for py, px in peaks:
    notch *= 1 - np.exp(-((YY - py) ** 2 + (XX - px) ** 2) / (2 * 3 ** 2))
restored = np.real(np.fft.ifft2(np.fft.ifftshift(Fc * notch)))
fig, axes = plt.subplots(1, 4, figsize=(21, 5))
for ax, im, title in zip(axes, [corrupted, power, power * notch, restored],
                         [f"periodic interference ({cv2.PSNR(cam, corrupted.astype(np.uint8)):.1f} dB)", "spectrum: two bright spikes",
                          "spectrum after notch filter", f"restored ({cv2.PSNR(cam, np.clip(restored, 0, 255).astype(np.uint8)):.1f} dB)"]):
    ax.imshow(im, cmap="gray" if "spectrum" not in title else "magma"); ax.set_title(title); ax.axis("off")
plt.suptitle("Figure 5.3 - Removing periodic noise in the frequency domain", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: low-pass cut-off

In [ ]:
def fft_demo(radius=40, mode="Gaussian low-pass"):
    Hm = {"ideal low-pass": (R <= radius).astype(float), "Gaussian low-pass": np.exp(-R ** 2 / (2 * radius ** 2)),
          "Gaussian high-pass": 1 - np.exp(-R ** 2 / (2 * radius ** 2))}[mode]
    out = np.real(np.fft.ifft2(np.fft.ifftshift(F * Hm)))
    kept = (np.abs(F * Hm) ** 2).sum() / (np.abs(F) ** 2).sum()
    show(np.log1p(np.abs(F * Hm)), out if "high" not in mode else np.abs(out),
         titles=[f"{mode}, radius {radius}", f"result ({kept:.1%} of the energy kept)"], figsize=(12, 6))


interact(fft_demo, radius=widgets.IntSlider(value=40, min=2, max=200), mode=["Gaussian low-pass", "ideal low-pass", "Gaussian high-pass"])

## 6. Mathematical morphology

Morphology processes **shapes** using a small binary shape called the **structuring element** $B$ (a disc, square, line...).
For a binary image $A$ (object = 1):

| Operation | Definition | Effect |
|---|---|---|
| **erosion** $A \ominus B$ | pixels where $B$ fits *entirely inside* $A$ | shrinks objects, removes small specks and thin bridges |
| **dilation** $A \oplus B$ | pixels where $B$ *touches* $A$ | grows objects, fills small gaps and holes |
| **opening** $(A \ominus B) \oplus B$ | erode, then dilate | removes objects smaller than $B$, keeps the rest (almost) unchanged |
| **closing** $(A \oplus B) \ominus B$ | dilate, then erode | fills holes and gaps smaller than $B$ |
| **gradient** $(A \oplus B) - (A \ominus B)$ | | outline of objects |
| **top-hat** $A - (A \circ B)$ | image minus its opening | small bright details on an uneven background |

On greyscale images, erosion becomes a local **minimum** and dilation a local **maximum**.

In [ ]:
A = np.zeros((12, 16), np.uint8)
A[2:9, 2:8] = 1; A[4:7, 8:13] = 1; A[5, 12:15] = 1; A[9, 3] = 1; A[3, 13] = 1; A[5, 4] = 0
B = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
ops = {"input A (with a hole, a bridge and specks)": A, "erosion": cv2.erode(A, B), "dilation": cv2.dilate(A, B),
       "opening (erode, dilate)": cv2.morphologyEx(A, cv2.MORPH_OPEN, B), "closing (dilate, erode)": cv2.morphologyEx(A, cv2.MORPH_CLOSE, B),
       "gradient (dilate - erode)": cv2.morphologyEx(A, cv2.MORPH_GRADIENT, B)}
fig, axes = plt.subplots(1, 6, figsize=(24, 3.6))
for ax, (name, M) in zip(axes, ops.items()):
    ax.imshow(M, cmap="Blues", vmin=0, vmax=1.3)
    ax.set_xticks(np.arange(-0.5, 16, 1), minor=True); ax.set_yticks(np.arange(-0.5, 12, 1), minor=True)
    ax.grid(which="minor", color="#dadce0", lw=0.6); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(name, fontsize=9)
plt.suptitle("Figure 6.1 - Binary morphology with a 3x3 square structuring element, pixel by pixel", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
text_bin = cv2.adaptiveThreshold(sudoku, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 25, 15)   # adaptive: uneven light
noisy_bin = text_bin.copy()
m = rng.random(noisy_bin.shape)
noisy_bin[m < 0.01] = 255; noisy_bin[m > 0.99] = 0
se = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))       # smaller than the thin grid lines, larger than the specks
cleaned = cv2.morphologyEx(cv2.morphologyEx(noisy_bin, cv2.MORPH_OPEN, se), cv2.MORPH_CLOSE, se)
tophat = cv2.morphologyEx(sudoku, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
skel = skeletonize(cleaned > 0)
show(noisy_bin, cleaned, tophat, skel.astype(np.uint8) * 255,
     titles=["noisy binary (adaptive threshold + 1% specks)", "open then close: specks and holes gone", "black-hat: dark print, uneven light removed",
             "skeleton: 1-pixel-wide centre lines"], figsize=(22, 5.5))

### Interactive: structuring elements and operations

In [ ]:
_, coins_bin = cv2.threshold(coins, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)


def morph_demo(operation="opening", shape="ellipse", size=7, image="coins"):
    src = {"coins": coins_bin, "sudoku text": noisy_bin}[image]
    se_ = cv2.getStructuringElement({"ellipse": cv2.MORPH_ELLIPSE, "rectangle": cv2.MORPH_RECT, "cross": cv2.MORPH_CROSS}[shape], (size, size))
    op = {"erosion": cv2.MORPH_ERODE, "dilation": cv2.MORPH_DILATE, "opening": cv2.MORPH_OPEN, "closing": cv2.MORPH_CLOSE,
          "gradient": cv2.MORPH_GRADIENT}[operation]
    out = cv2.morphologyEx(src, op, se_)
    n_in, n_out = cv2.connectedComponents(src)[0] - 1, cv2.connectedComponents(out)[0] - 1
    show(src, out, titles=[f"input: {n_in} blobs", f"{operation} ({shape} {size}): {n_out} blobs"], figsize=(12, 5))


interact(morph_demo, operation=["opening", "closing", "erosion", "dilation", "gradient"], shape=["ellipse", "rectangle", "cross"],
         size=widgets.IntSlider(value=7, min=1, max=31, step=2), image=["coins", "sudoku text"])

### From kernels to CNNs
Every filter in this notebook uses a kernel **designed by a person**: Gaussian for noise, Sobel for edges, a disc for morphology. In Week 6 a
convolutional neural network will contain dozens of layers of exactly these operations (convolution, a non-linearity, max-pooling, which is a
greyscale dilation), but with the kernel values **learned from examples**. Keep Figure 4.2 in mind: the first layer of a trained CNN learns
filters that look remarkably like Sobel and Gabor kernels.

## 7. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Correlation by hand
Implement 2D correlation of a greyscale `float32` image with an odd-sized kernel using **zero padding** and NumPy only (no `cv2.filter2D`, no `scipy`). Output has the same size as the input. Hint: loop over the *kernel* entries, not over pixels.

In [ ]:
def correlate2d(img, kernel):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    img = cam[100:180, 150:250].astype(np.float32)
    for k in (np.ones((3, 3), np.float32) / 9, np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32),
              rng.normal(size=(5, 5)).astype(np.float32)):
        got = np.asarray(correlate2d(img, k))
        ref = cv2.filter2D(img, -1, k, borderType=cv2.BORDER_CONSTANT)
        assert got.shape == img.shape, f"output shape {got.shape} should equal input shape {img.shape}"
        err = np.abs(got - ref).max()
        assert err < 1e-3, f"differs from cv2.filter2D (zero border) by {err:.3g}. Did you flip the kernel (that would be convolution)?"


check("E1 correlate2d", _test_e1)

### E2. PSNR
Implement PSNR (dB) for two `uint8` images. Return `float('inf')` for identical images.

In [ ]:
def psnr(a, b):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    assert psnr(cam, cam) == float("inf"), "identical images should give infinite PSNR"
    for im in noisy.values():
        assert abs(psnr(clean, im) - cv2.PSNR(clean, im)) < 1e-6, "differs from cv2.PSNR: compute the MSE in float, not uint8!"


check("E2 psnr", _test_e2)

### E3. Build a Gaussian kernel
Return a normalised 2D Gaussian kernel of size `size × size` (odd) and standard deviation `sigma`, using its separability. Do not use `cv2.getGaussianKernel`.

In [ ]:
def gaussian_kernel(size, sigma):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    for size, sigma in [(5, 1.0), (9, 2.0), (15, 3.5)]:
        k = np.asarray(gaussian_kernel(size, sigma), float)
        assert k.shape == (size, size), f"expected ({size}, {size}), got {k.shape}"
        assert abs(k.sum() - 1) < 1e-9, f"kernel must sum to 1, got {k.sum():.6f}"
        g = cv2.getGaussianKernel(size, sigma)
        assert np.allclose(k, g @ g.T, atol=1e-9), f"size {size}, sigma {sigma}: values differ from the Gaussian formula"


check("E3 gaussian_kernel", _test_e3)

### E4. Count the coins (no self-check)
Using `coins`: smooth, threshold (Otsu), clean with opening/closing, then count the objects with `cv2.connectedComponentsWithStats`, ignoring blobs
smaller than 100 pixels. Some coins touch or have dark centres: which morphological step fixes which problem? (Week 4B will split touching coins with watershed.)

In [ ]:
# E4: your code here

## 8. Challenge: clean a real sensor image

Capture 10 frames from your webcam in a dark room (high gain = strong noise), or use `noisy["Poisson (low light)"]`.
1. Compare frame averaging (Week 3A), Gaussian, bilateral and non-local means by PSNR (use the average of 64 frames as the reference) and by run time.
2. Which method would you deploy on a 30 frames-per-second inspection line, and which for a single high-quality image? Justify with your numbers.

In [ ]:
# Your challenge code here

## 9. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'An image has salt-and-pepper noise. Which filter removes it best while keeping edges sharp?',
        "options": [
            'Mean 5x5',
            'Gaussian',
            'Median',
            'Laplacian',
        ],
        "answer": 'e984a6fffb',
        "explain": 'Isolated extreme pixels are never the median of their window.',
    },
    {
        "q": 'Why is the 2D Gaussian filter usually applied as two 1D passes?',
        "options": [
            'It gives a sharper result',
            'It is separable, so the cost drops from k^2 to 2k operations per pixel',
            'OpenCV requires it',
            'It avoids border effects',
        ],
        "answer": '787a568d77',
        "explain": 'h(x, y) = h(x) h(y).',
    },
    {
        "q": 'What does the Sobel operator approximate?',
        "options": [
            'The image mean',
            'The first derivative (gradient) combined with smoothing',
            'The second derivative only',
            'A Fourier transform',
        ],
        "answer": '9291145e10',
        "explain": 'Difference in one direction, [1 2 1] smoothing in the other.',
    },
    {
        "q": 'In the Fourier spectrum, where does fine detail (edges, noise) appear?',
        "options": [
            'At the centre',
            'Far from the centre (high frequencies)',
            'Only on the horizontal axis',
            'It does not appear',
        ],
        "answer": '69719a51b8',
        "explain": 'The centre is the zero frequency (mean brightness).',
    },
    {
        "q": 'Why does an ideal (sharp-edged) low-pass mask cause ringing?',
        "options": [
            'It removes too little',
            'A sharp cut-off in frequency corresponds to an oscillating (sinc) kernel in space',
            'It is not linear',
            'Because of the border mode',
        ],
        "answer": 'db197134b5',
        "explain": 'Smooth masks such as a Gaussian have smooth, non-oscillating kernels.',
    },
    {
        "q": 'Which morphological operation removes small white specks while keeping larger objects about the same size?',
        "options": [
            'Dilation',
            'Closing',
            'Opening',
            'Gradient',
        ],
        "answer": '9d5e373a75',
        "explain": 'Erosion removes the specks, and the following dilation restores the survivors.',
    },
    {
        "q": 'A binary mask has small black holes inside the objects. Which operation fills them?',
        "options": [
            'Opening',
            'Closing',
            'Erosion',
            'Top-hat',
        ],
        "answer": '57d3d31677',
        "explain": 'Dilation fills the holes, and the following erosion restores the outline.',
    },
], title='Week 03B quiz')

## 10. Exit ticket (reflection)
1. Why must you smooth before differentiating? Illustrate with Figure 4.1.
2. A scanner adds diagonal stripes to every image. Would you remove them in the spatial or the frequency domain? Why?
3. Give an inspection example where you would use opening, and another where you would use closing.

## 11. What's new (2025–26)

> - **Learned denoisers** (e.g. DnCNN, Restormer and diffusion-based restoration) outperform classical filters on hard noise, but need training data
>   and GPUs. Non-local means and bilateral filters remain the go-to for fast, predictable pipelines.
> - Many industrial cameras now perform **on-sensor or in-FPGA filtering** (denoise, flat-field, sharpening) before the image reaches the PC.
> - **Morphological layers** and learnable structuring elements appear in research networks, and max-pooling in every CNN is a greyscale dilation.

## 12. References and further exploration

**Textbooks**
- Gonzalez & Woods (2018). *Digital Image Processing*, 4th ed., ch. 3 (spatial filtering), ch. 4 (frequency domain), ch. 5 (restoration), ch. 9 (morphology).
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §3.2–3.4 (linear/non-linear filtering, Fourier). [szeliski.org/Book](https://szeliski.org/Book/)
- Soille, P. (2003). *Morphological Image Analysis*, 2nd ed. Springer.
- Serra, J. (1982). *Image Analysis and Mathematical Morphology.* Academic Press.

**Papers**
- Tomasi, C. & Manduchi, R. (1998). *Bilateral filtering for gray and color images.* ICCV. [doi:10.1109/ICCV.1998.710815](https://doi.org/10.1109/ICCV.1998.710815)
- Buades, A., Coll, B. & Morel, J.-M. (2005). *A non-local algorithm for image denoising.* CVPR. [doi:10.1109/CVPR.2005.38](https://doi.org/10.1109/CVPR.2005.38)
- Cooley, J. W. & Tukey, J. W. (1965). *An algorithm for the machine calculation of complex Fourier series* (the FFT). Math. Comp. 19. [doi:10.2307/2003354](https://doi.org/10.2307/2003354)

**Interactive and video**
- Victor Powell: *Image Kernels Explained Visually* (setosa.io). [setosa.io/ev/image-kernels](https://setosa.io/ev/image-kernels/)
- 3Blue1Brown: *But what is a convolution?* [youtube.com/watch?v=KuXjwB4LzSA](https://www.youtube.com/watch?v=KuXjwB4LzSA)
- 3Blue1Brown: *But what is the Fourier Transform? A visual introduction.* [youtube.com/watch?v=spUNpyF58BY](https://www.youtube.com/watch?v=spUNpyF58BY)
- OpenCV tutorials: *Smoothing images*, *Image gradients*, *Fourier transform*, *Morphological transformations*. [docs.opencv.org](https://docs.opencv.org/4.x/d2/d96/tutorial_py_table_of_contents_imgproc.html)

**Images:** `camera`, `coins` from scikit-image (public domain); `sudoku.png` from the OpenCV sample data.

## 13. Key takeaways
- **Convolution/correlation**: weighted sums over a window; mind the **border mode**; Gaussians are **separable**.
- Match the filter to the **noise**: Gaussian/bilateral/NLM for Gaussian noise, **median** for salt-and-pepper. Measure with **PSNR**.
- **Derivatives** (Sobel, Laplacian) find edges but amplify noise: **smooth first**. Unsharp masking sharpens.
- The **Fourier transform** turns convolution into multiplication; low-pass = blur, high-pass = edges, **notch** filters remove periodic noise.
- **Morphology** cleans binary masks: opening removes specks, closing fills holes, gradient outlines, top-hat corrects backgrounds, skeletons give centre lines.

**Next week (04A):** segmentation by thresholding (global, Otsu, adaptive), edge detection (Canny) and the Hough transform.